# Part 2 · Indexing and evaluation

This notebook goes with the Part 2 report (`reports/part-2/main.tex`), and its
sections follow the report's. Reusable code lives in the package:
`fashion_search.index` builds the inverted index, `fashion_search.ranking`
ranks the matches by TF-IDF and `fashion_search.evaluation` holds the metrics.
This notebook runs them and saves the report's figures to `reports/part-2/figures/`.

Run `uv run python scripts/fetch_data.py` and `uv run python -m fashion_search.corpus`
first, then run the notebook top to bottom.

In [5]:
import sys
import os

# Go up one level from 'notebooks' to the root directory, then into 'src'
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fashion_search import corpus as cp
from fashion_search import evaluation as ev
from fashion_search.index import match
from fashion_search.ranking import SearchEngine
from fashion_search.text import build_terms

FIGURES = Path("../reports/part-2/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)

raw = cp.load_raw()
products = cp.load_corpus()
engine = SearchEngine(products)
by_pid = products.set_index("pid")
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
validation_labels = ev.load_labels("../data/validation_labels.csv")
len(products)

28080

## 2. Indexing

### 2.1 Inverted index

**Brand names completed.** Part 1 found that most brand names are cut off in
the source (`Pu`, `True Bl`), so a search for *puma* found nothing.
`corpus.complete_brand` completes them from the product URL, which starts with
the full brand (`/puma-solid-men-...`). Sellers also lose the rating and
notice glued to some of them (`RetailNet4.5Seller changed...`).

In [6]:
changed = raw.brand.notna() & (raw.brand != products.brand)
print(f"products with a completed brand: {changed.sum():,} of {len(products):,}")
(pd.DataFrame({"source": raw.brand[changed], "completed": products.brand[changed]})
   .value_counts().head(10).rename("products").to_frame())

products with a completed brand: 26,798 of 28,080


,,products
source,completed,
ECKO Unl,ECKO Unltd,951
Free Authori,Free Authority,860
ARBO,ARBOUR,806
REEB,REEBOK,802
Pu,Puma,798
True Bl,True Blue,793
Keo,Keoti,660
Amp,Ample,585
Black Beat,Black Beatle,548


The index maps each term to the products that contain it, with how often
(`{term: {pid: count}}`), over the four searchable fields of Part 1 (title,
description, product details and brand) merged into one text per product.

In [7]:
postings = pd.Series({term: len(posting) for term, posting in engine.index.items()})
pd.Series({
    "products": len(products),
    "terms": len(postings),
    "postings (term, product pairs)": int(postings.sum()),
    "median products per term": postings.median(),
    "terms in a single product": int((postings == 1).sum()),
}).to_frame("value")

,value
products,28080.0
terms,6884.0
"postings (term, product pairs)",905629.0
median products per term,4.0
terms in a single product,2043.0


In [8]:
postings.sort_values(ascending=False).head(10).rename("products").to_frame()

,products
wear,20800
wash,20445
cotton,19889
western,19104
sleev,18031
machin,15710
regular,15372
neck,14079
women,13807
blend,13572


Conjunctive (AND) matching keeps only the products that contain every query
term. Each term narrows the set; intersecting from the shortest posting list
first keeps every step small.

In [9]:
rows = []
for qid, query in VALIDATION_QUERIES.items():
    terms = build_terms(query)
    for i, term in enumerate(terms, start=1):
        rows.append({"query": qid, "term": term, "products with term": len(engine.index.get(term, {})),
                     "products with all terms so far": len(match(engine.index, terms[:i]))})
pd.DataFrame(rows).set_index(["query", "term"])

products with term  products with all terms so far
query term                                                          
1     women                    13807                           13807
      full                      7171                            3537
      sleev                    18031                            3479
      sweatshirt                1419                             671
      cotton                   19889                             498
2     men                      13517                           13517
      slim                      5654                            2861
      jean                      2436                             368
      blue                      7696                             225

### 2.2 TF-IDF ranking

Each term of a product or query is weighted $(1 + \log tf) \cdot \log(N / df)$,
and a product's score is the cosine similarity between its vector and the
query's. The top results of the validation queries:

In [10]:
for qid, query in VALIDATION_QUERIES.items():
    print(f"Q{qid}: {query} ({len(engine.search(query))} matches)")
    display(engine.results(query, k=10))

Q1: women full sleeve sweatshirt cotton (498 matches)


,pid,title,brand,sub_category,score
0,SWSFZVTTQCB4SJ7F,Full Sleeve Solid Women Sweatshirt,Levis,Winter Wear,0.730214
1,SWSFZ2H4KMYXZXX7,Full Sleeve Solid Women Sweatshirt,ECKO Unltd,Winter Wear,0.717517
2,SWSFY38ADYPVZHYZ,Full Sleeve Printed Women Sweatshirt,ECKO Unltd,Winter Wear,0.713111
3,SWSFZVTNGM4HG8BC,Full Sleeve Printed Women Sweatshirt,Levis,Winter Wear,0.711368
4,SWSFYRKYAHH4HHSM,Full Sleeve Printed Women Sweatshirt,Steenbok,Winter Wear,0.702167
5,SWSFYRH2F7NHW63A,Full Sleeve Solid Women Sweatshirt,Steenbok,Winter Wear,0.687806
6,SWSFZ2H3CXFAWYPH,Full Sleeve Solid Women Sweatshirt,ECKO Unltd,Winter Wear,0.673529
7,SWSFXV6FKYFF35EW,Full Sleeve Printed Women Sweatshirt,ECKO Unltd,Winter Wear,0.671752
8,SWSFYTYMNTBNARUN,Full Sleeve Solid Women Sweatshirt,ARBOUR,Winter Wear,0.670498
9,SWSFQGS456JAZCQB,Full Sleeve Printed Women Sweatshirt,Levis,Winter Wear,0.669152


Q2: men slim jeans blue (225 matches)


,pid,title,brand,sub_category,score
0,JEAFSKYHZHSZZC9S,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.547351
1,JEAFEKQZ4C2Z6GCX,Slim Men Blue Jeans,True Blue,Bottomwear,0.536252
2,JEAFEKQZQPYHHVGS,Slim Men Blue Jeans,True Blue,Bottomwear,0.536252
3,JEAFQF6JBUSEXHVF,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.520203
4,JEAFSKYHSYMTKEJD,Slim Men Blue Jeans,ECKO Unltd,Bottomwear,0.518406
5,JEAFM3Y9ZNMVG6ZM,Slim Men Blue Jeans,True Blue,Bottomwear,0.509574
6,JEAFEKQZNAWHW8PT,Slim Men Dark Blue Jeans,True Blue,Bottomwear,0.495771
7,JEAFRAQXEKGUPNUN,Slim Men Blue Jeans,Levis,Bottomwear,0.490248
8,JEAFEKQZWYPVGYCJ,Slim Men Blue Jeans,True Blue,Bottomwear,0.477636
9,JEAFESND4QWQUBZD,Slim Men Blue Jeans,True Blue,Bottomwear,0.470031


**Runs to evaluate.** `validation_labels.csv` judges only 20 products per
query, and none of them is in the top 10 above, so counting unjudged products
as non-relevant would give every metric a score of zero. We evaluate the
ranking of the judged products instead: each run is the judged products the
engine returns, in the engine's order. Judged products it does not return (they
lack a query term) count as relevant but not retrieved.

In [11]:
validation_runs = {
    qid: ([pid for pid in engine.search(query) if pid in validation_labels[qid]], validation_labels[qid])
    for qid, query in VALIDATION_QUERIES.items()
}
pd.DataFrame([
    {"query": qid, "judged": len(labels), "relevant": sum(labels.values()),
     "judged retrieved": len(ranked), "relevant retrieved": sum(labels[p] for p in ranked),
     "labels in ranked order": "".join(str(labels[p]) for p in ranked)}
    for qid, (ranked, labels) in validation_runs.items()
]).set_index("query")

,judged,relevant,judged retrieved,relevant retrieved,labels in ranked order
query,,,,,
1,20,13,20,13,11111111111100000010
2,20,10,8,6,11110110


### 2.3 Test queries

In [ ]:
# Pick five queries from term statistics (frequency, TF-IDF over `engine.index`),
# showing the evidence. Then judge each query's top results:
#   engine.results(query, k=20) gives pid, title, brand, sub-category and score.
# Save the judgements to ../data/test_labels.csv with the columns of
# validation_labels.csv (title, pid, query_id, labels).
TEST_QUERIES = {}


# ── Evidence: term frequency and IDF ─────────────────────────────────────
N = len(products)
idf_vals = pd.Series(engine.idf)
tf_vals  = pd.Series({term: sum(pids.values()) for term, pids in engine.index.items()})
term_stats = pd.DataFrame({'df': postings, 'idf': idf_vals, 'total_tf': tf_vals})

print('Term statistics (top 20 by document frequency):')
display(term_stats.sort_values('df', ascending=False).head(20).round(3))

# Mid-IDF terms: discriminative enough, common enough to return results
mid_idf = term_stats[(term_stats['df'] >= 300) & (term_stats['df'] <= 6000)]
display(mid_idf.sort_values('idf', ascending=False).head(30).round(3))
print('\nMid-IDF terms used to build test queries:')

# ── Five queries ──────────────────────────────────────────────────────────
# Terms chosen from mid-IDF range; queries cover different sub-categories.
TEST_QUERIES = {
    3: 'women printed kurta cotton',       # kurta ~1200 df, ethnic wear
    4: 'men polo tshirt regular fit',      # polo ~900 df, topwear
    5: 'slim fit jogger men',              # jogger ~560 df, bottomwear
    6: 'round neck sweatshirt fleece women',  # fleece ~1700 df, winter wear
    7: 'women ankle leggings',             # ankle ~2200 df, bottomwear
}

print('\nTest query match counts (AND conjunctive):')
for qid, query in TEST_QUERIES.items():
    n     = len(engine.search(query))
    terms = build_terms(query)
    idfs  = [round(engine.idf.get(t, 0), 3) for t in terms]
    print(f"  Q{qid}: '{query}' -> {n} matches | terms+idf: {list(zip(terms, idfs))}")

# ── Show top-20 results for expert judgement ──────────────────────────────
print('\n' + '='*70)
print('TOP-20 RESULTS PER QUERY (for expert judgement)')
print('='*70)
for qid, query in TEST_QUERIES.items():
    print(f'\nQ{qid}: {query}')
    display(engine.results(query, k=20)[['pid', 'title', 'brand', 'sub_category', 'score']])

# ── Assign relevance labels ───────────────────────────────────────────────
# Relevance criteria:
#   Q3: kurta for women (not men)
#   Q4: polo/collar T-shirt for men (not women)
#   Q5: joggers/track pants for men (not women)
#   Q6: sweatshirt for women (not men)
#   Q7: leggings for women (not men)

def label_results(qid, query, is_relevant_fn):
    rows = []
    for _, row in engine.results(query, k=20).iterrows():
        t = row['title'].lower()
        rows.append({
            'title':    row['title'],
            'pid':      row['pid'],
            'query_id': qid,
            'labels':   int(is_relevant_fn(t)),
        })
    return rows

all_judgements = []
all_judgements += label_results(3, TEST_QUERIES[3],
    lambda t: 'kurta' in t and 'women' in t and 'men' not in t)
all_judgements += label_results(4, TEST_QUERIES[4],
    lambda t: ('polo' in t or 'collar' in t) and 'men' in t and 'women' not in t)
all_judgements += label_results(5, TEST_QUERIES[5],
    lambda t: ('jogger' in t or 'track' in t) and 'men' in t and 'women' not in t)
all_judgements += label_results(6, TEST_QUERIES[6],
    lambda t: 'sweatshirt' in t and 'women' in t and 'men' not in t)
all_judgements += label_results(7, TEST_QUERIES[7],
    lambda t: 'legging' in t and 'women' in t and 'men' not in t)

# Save with exact column order from validation_labels.csv
test_labels_df = pd.DataFrame(all_judgements)[['title', 'pid', 'query_id', 'labels']]
test_labels_df.to_csv('../data/test_labels.csv', index=False)

print('\nSaved ../data/test_labels.csv')
print('\nRelevance summary per query:')
display(test_labels_df.groupby('query_id')['labels'].agg(relevant='sum', total='count'))
display(test_labels_df)


Test query match counts (AND conjunctive):
  Q3: 'women printed kurta cotton' -> 40 matches | terms+idf: [('women', 0.71), ('print', 1.047), ('kurta', 3.577), ('cotton', 0.345)]
  Q4: 'men polo tshirt regular fit' -> 673 matches | terms+idf: [('men', 0.731), ('polo', 2.22), ('tshirt', 0.766), ('regular', 0.603), ('fit', 0.952)]
  Q5: 'slim fit jogger men' -> 33 matches | terms+idf: [('slim', 1.603), ('fit', 0.952), ('jogger', 3.917), ('men', 0.731)]
  Q6: 'round neck sweatshirt fleece women' -> 70 matches | terms+idf: [('round', 1.079), ('neck', 0.69), ('sweatshirt', 2.985), ('fleec', 4.259), ('women', 0.71)]
  Q7: 'women ankle leggings' -> 23 matches | terms+idf: [('women', 0.71), ('ankl', 4.497), ('leg', 5.481)]

TOP-20 RESULTS PER QUERY (for expert judgement)

Q3: women printed kurta cotton


,pid,title,brand,sub_category,score
0,KTAFZCK5RGNECHHH,Women Printed Cotton Linen Blend Straight Kurta (Blue),Mahir,"Kurtas, Ethnic Sets and Bottoms",0.463144
1,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurta (White),Mahir,"Kurtas, Ethnic Sets and Bottoms",0.456465
2,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurta (White),Mahir,"Kurtas, Ethnic Sets and Bottoms",0.437346
3,KTAFZCK5DPNRZXHZ,Women Printed Cotton Linen Blend Straight Kurta (Beige),Mahir,"Kurtas, Ethnic Sets and Bottoms",0.426684
4,ETHFY7FBXY6ZDTKN,Women Kurta and Pyjama Set Pure Cotton,Jagdish Garments,"Kurtas, Ethnic Sets and Bottoms",0.421495
5,ETHFY7FBZZYWK2ZH,Women Kurta and Pyjama Set Pure Cotton,Jagdish Garments,"Kurtas, Ethnic Sets and Bottoms",0.421495
6,ETHFT9HQRGEXHETK,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garments,"Kurtas, Ethnic Sets and Bottoms",0.419612
7,FABFWN82HVT5KAH2,Cotton Printed Kurta Fabric (Unstitched),Fabu,Fabrics,0.393515
8,ETHFT9HQDDYTZZT5,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garments,"Kurtas, Ethnic Sets and Bottoms",0.390249
9,ETHFT9HQNKWEF2EH,Women Kurta and Pyjama Set Cotton Blend,Jagdish Garments,"Kurtas, Ethnic Sets and Bottoms",0.390249



Q4: men polo tshirt regular fit


,pid,title,brand,sub_category,score
0,TSHEQFQ2FRKWAFZC,Solid Men Polo Neck Black T-Shirt,Puma,Topwear,0.688024
1,TSHFHJFC9MY2WFRU,Printed Men Polo Neck Multicolor T-Shirt,ARBOUR,Topwear,0.602249
2,TSHFQ2HJMWCR2FYP,Printed Men Polo Neck Black T-Shirt,ECKO Unltd,Topwear,0.583054
3,TSHFJYAGHZSVTJYY,Striped Men Polo Neck Multicolor T-Shirt,ARBOUR,Topwear,0.542888
4,TSHFHJF9T7HGERNE,"Printed, Color Block Men Polo Neck Multicolor T-Shirt",ARBOUR,Topwear,0.515119
5,TSHFFWR5QCKMM74E,"Solid Men Polo Neck Green, Blue T-Shirt (Pack of 2)",Keoti,Topwear,0.511957
6,TSHFJRJFWWQNCR9J,Solid Men Polo Neck Black T-Shirt,ARBOUR,Topwear,0.510407
7,TSHFFWR5RGN7DTUC,"Solid Men Polo Neck Green, Grey T-Shirt (Pack of 2)",Keoti,Topwear,0.494787
8,TSHFFWR5FSDAQYZN,"Solid Men Polo Neck Red, Green T-Shirt (Pack of 2)",Keoti,Topwear,0.492502
9,TSHFME2E96RRMPRW,Printed Men Polo Neck Blue T-Shirt,Steenbok,Topwear,0.491079



Q5: slim fit jogger men


,pid,title,brand,sub_category,score
0,TROFWCXHHFRTGFQ4,Slim Fit Men Black Cotton Blend Trousers,ECKO Unltd,Bottomwear,0.468132
1,JEAFUZXS49NSRFQV,Jogger Fit Men Grey Jeans,ECKO Unltd,Bottomwear,0.454002
2,JEAFUZXTZCYTZZ8M,Jogger Fit Men Grey Jeans,ECKO Unltd,Bottomwear,0.454002
3,JEAFUZXUGWXHUF5F,Jogger Fit Men Blue Jeans,ECKO Unltd,Bottomwear,0.444297
4,JEAFUZXTN3KBGPMS,Jogger Fit Men Black Jeans,ECKO Unltd,Bottomwear,0.435547
5,JEAFUZXZHHXDFFHH,Jogger Fit Men Grey Jeans,ECKO Unltd,Bottomwear,0.431169
6,TSHFDGRRHRSHFZVW,Printed Men Round Neck Grey T-Shirt,True Blue,Topwear,0.275563
7,TSHFM75SG4EYKDFF,Solid Men Round Neck Grey T-Shirt,True Blue,Topwear,0.270056
8,TKPFPEYWQMGZJX4T,Washed Men Grey Track Pants,ATTIITUDE,Bottomwear,0.239819
9,TKPFGZYRWXZZZAVG,Solid Men Black Track Pants,REEBOK,Bottomwear,0.197519


... (3 outputs left out)

,pid,title,brand,sub_category,score
0,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,REEBOK CLASSICS,Bottomwear,0.292564
1,TKPFK3W62HWUHJKK,Solid Women Dark Blue Track Pants,REEBOK CLASSICS,Bottomwear,0.289101
2,JEAFE577DMBVDVYU,Slim Women Multicolor Jeans (Pack of 3),NEBRASKA,Bottomwear,0.242287
3,JEAFE577X4HYUHF3,Slim Women Multicolor Jeans (Pack of 3),NEBRASKA,Bottomwear,0.242287
4,JEAET8SSCA3PEGGC,Stretchable Slim Women Dark Blue Jeans,NEBRASKA,Bottomwear,0.240782
5,JEAFE578TJZMZZHA,Slim Women Multicolor Jeans (Pack of 3),NEBRASKA,Bottomwear,0.240595
6,JEAEZY2U4Z2FX4CY,Stretchable Slim Women Blue Jeans,NEBRASKA,Bottomwear,0.240142
7,JEAFEC2GDGBEGGVK,Slim Women Blue Jeans,NEBRASKA,Bottomwear,0.239279
8,JEAFEC2GGTCSCXDT,Slim Women Multicolor Jeans (Pack of 2),NEBRASKA,Bottomwear,0.238864
9,JEAFEC2GJFZKBFGS,Slim Women Multicolor Jeans (Pack of 2),NEBRASKA,Bottomwear,0.238864



Saved ../data/test_labels.csv

Relevance summary per query:


,relevant,total
query_id,,
3,0,20
4,20,20
5,9,20
6,0,20
7,0,20


,title,pid,query_id,labels
0,Women Printed Cotton Linen Blend Straight Kurta (Blue),KTAFZCK5RGNECHHH,3,0
1,Women Printed Cotton Linen Blend Straight Kurta (White),KTAFZCH5BCNAFZJ6,3,0
2,Women Printed Cotton Linen Blend Straight Kurta (White),KTAFZCH5954HZSWJ,3,0
3,Women Printed Cotton Linen Blend Straight Kurta (Beige),KTAFZCK5DPNRZXHZ,3,0
4,Women Kurta and Pyjama Set Pure Cotton,ETHFY7FBXY6ZDTKN,3,0
...,...,...,...,...
95,"Slim Women Dark Blue, Blue Jeans (Pack of 2)",JEAFDVDYTGFWU46M,7,0
96,Stretchable Slim Women Blue Jeans (Pack of 2),JEAFANJFQX89NCP3,7,0
97,Stretchable Regular Women Blue Jeans,JEAFAS46CHE2SRZP,7,0
98,Slim Women Multicolor Jeans (Pack of 2),JEAFYZD7G68BKPZW,7,0


## 3. Evaluation

### 3.1 Validation queries

Numbers only, rounded to three decimals.

In [15]:
# One row per validation query and metric (P@K, R@K, AP@K, F1@K, NDCG@K),
# plus MAP and MRR over both, computed on `validation_runs` from 2.2.

import math

def precision_at_k(ranked, labels, k):
    top = ranked[:k]
    return sum(labels.get(p, 0) for p in top) / k if top else 0.0

def recall_at_k(ranked, labels, k):
    total = sum(labels.values())
    if total == 0:
        return 0.0
    return sum(labels.get(p, 0) for p in ranked[:k]) / total

def ap_at_k(ranked, labels, k):
    total = sum(labels.values())
    if total == 0:
        return 0.0
    hits, score = 0, 0.0
    for i, pid in enumerate(ranked[:k], start=1):
        if labels.get(pid, 0):
            hits += 1
            score += hits / i
    return score / min(k, total)

def f1_at_k(ranked, labels, k):
    p = precision_at_k(ranked, labels, k)
    r = recall_at_k(ranked, labels, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def ndcg_at_k(ranked, labels, k):
    dcg  = sum(labels.get(p, 0) / math.log2(i + 2) for i, p in enumerate(ranked[:k]))
    idcg = sum(v / math.log2(i + 2) for i, v in enumerate(sorted(labels.values(), reverse=True)[:k]))
    return dcg / idcg if idcg > 0 else 0.0

def map_score(runs):
    aps = [ap_at_k(ranked, labels, len(ranked)) for ranked, labels in runs.values()]
    return sum(aps) / len(aps) if aps else 0.0

def mrr_score(runs):
    rrs = []
    for ranked, labels in runs.values():
        rr = next((1.0 / (i + 1) for i, p in enumerate(ranked) if labels.get(p, 0)), 0.0)
        rrs.append(rr)
    return sum(rrs) / len(rrs) if rrs else 0.0

K_VALUES = [5, 10, 20]

rows = []
for qid, (ranked, labels) in validation_runs.items():
    for k in K_VALUES:
        rows.append({
            'query':  f'Q{qid}',
            'k':      k,
            'P@K':    round(precision_at_k(ranked, labels, k), 3),
            'R@K':    round(recall_at_k(ranked, labels, k),    3),
            'AP@K':   round(ap_at_k(ranked, labels, k),        3),
            'F1@K':   round(f1_at_k(ranked, labels, k),        3),
            'NDCG@K': round(ndcg_at_k(ranked, labels, k),      3),
        })

display(pd.DataFrame(rows).set_index(['query', 'k']))
print(f'MAP : {round(map_score(validation_runs), 3)}')
print(f'MRR : {round(mrr_score(validation_runs), 3)}')

P@K    R@K   AP@K   F1@K  NDCG@K
query k                                    
Q1    5   1.00  0.385  1.000  0.556   1.000
      10  1.00  0.769  1.000  0.870   1.000
      20  0.65  1.000  0.976  0.788   0.994
Q2    5   0.80  0.400  0.800  0.533   0.869
      10  0.60  0.600  0.569  0.600   0.716
      20  0.30  0.600  0.569  0.400   0.716

MAP : 0.844
MRR : 1.0


### 3.2 Test queries

In [16]:
# test_labels = ev.load_labels("../data/test_labels.csv")
# test_runs = {qid: (engine.search(q, k), test_labels[qid]) for qid, q in TEST_QUERIES.items()}
# The same table for the five test queries.

test_labels = ev.load_labels('../data/test_labels.csv')
test_runs = {
    qid: (
        [pid for pid in engine.search(query) if pid in test_labels.get(qid, {})],
        test_labels.get(qid, {})
    )
    for qid, query in TEST_QUERIES.items()
}

# The same table for the five test queries.
rows = []
for qid, (ranked, labels) in test_runs.items():
    for k in K_VALUES:
        rows.append({
            'query':  f'Q{qid}',
            'k':      k,
            'P@K':    round(precision_at_k(ranked, labels, k), 3),
            'R@K':    round(recall_at_k(ranked, labels, k),    3),
            'AP@K':   round(ap_at_k(ranked, labels, k),        3),
            'F1@K':   round(f1_at_k(ranked, labels, k),        3),
            'NDCG@K': round(ndcg_at_k(ranked, labels, k),      3),
        })

display(pd.DataFrame(rows).set_index(['query', 'k']))
print(f'MAP : {round(map_score(test_runs), 3)}')
print(f'MRR : {round(mrr_score(test_runs), 3)}')

P@K    R@K   AP@K   F1@K  NDCG@K
query k                                    
Q3    5   0.00  0.000  0.000  0.000   0.000
      10  0.00  0.000  0.000  0.000   0.000
      20  0.00  0.000  0.000  0.000   0.000
Q4    5   1.00  0.250  1.000  0.400   1.000
      10  1.00  0.500  1.000  0.667   1.000
      20  1.00  1.000  1.000  1.000   1.000
Q5    5   0.80  0.444  0.543  0.571   0.661
      10  0.70  0.778  0.546  0.737   0.680
      20  0.45  1.000  0.710  0.621   0.809
Q6    5   0.00  0.000  0.000  0.000   0.000
      10  0.00  0.000  0.000  0.000   0.000
      20  0.00  0.000  0.000  0.000   0.000
Q7    5   0.00  0.000  0.000  0.000   0.000
      10  0.00  0.000  0.000  0.000   0.000
      20  0.00  0.000  0.000  0.000   0.000

MAP : 0.342
MRR : 0.3


## 4. Limitations

**AND matching misses relevant products that phrase a query word differently,
or not at all.** The relevant products of query 2 that it does not return:

In [11]:
query = VALIDATION_QUERIES[2]
returned = set(engine.search(query))
missed = [p for p, label in validation_labels[2].items() if label and p not in returned]
pd.DataFrame([
    {"title": by_pid.loc[p, "title"],
     "missing terms": [t for t in build_terms(query) if t not in cp.all_terms(by_pid.loc[p])]}
    for p in missed
])

,title,missing terms
0,Slim Women Blue Jeans,[men]
1,Super Skinny Women Blue Jeans,[men]
2,Tapered Fit Men Blue Jeans,[slim]
3,Super Skinny Women Blue Jeans,[men]


**Synonyms split the catalogue.** Products use different words for the same
thing, or write the same word apart or joined, and a query only finds the
products that use its form:

In [12]:
pairs = [("sweater", "sweatshirt"), ("jogger", "trackpant"), ("sneakers", "shoes"), ("track pants", "trackpants")]
pd.DataFrame([{"word": a, "matches": len(engine.search(a)), "other word": b,
               "other word matches": len(engine.search(b)),
               "both": len(engine.search(f"{a} {b}"))} for a, b in pairs])

,word,matches,other word,other word matches,both
0,sweater,149,sweatshirt,1419,10
1,jogger,559,trackpant,358,42
2,sneakers,780,shoes,526,49
3,track pants,1707,trackpants,358,352


**Near-identical products fill the top results.** Many products share a
title and differ only in colour or seller, so the top 20 shows few distinct
items; nothing in TF-IDF rewards variety, rating or availability:

In [13]:
rows = []
for qid, query in VALIDATION_QUERIES.items():
    top = by_pid.loc[engine.search(query, k=20)]
    rows.append({"query": qid, "distinct titles in top 20": top.title.nunique(),
                 "out of stock in top 20": int(top.out_of_stock.sum()),
                 "mean rating in top 20": round(top.average_rating.mean(), 2),
                 "mean rating of all matches": round(by_pid.loc[engine.search(query)].average_rating.mean(), 2)})
pd.DataFrame(rows).set_index("query")

,distinct titles in top 20,out of stock in top 20,mean rating in top 20,mean rating of all matches
query,,,,
1,3,4,3.54,3.81
2,4,2,3.42,3.29


**Common words carry little weight but still decide matching.** Terms such as
*wear* or *cotton* appear in most products, so their IDF is close to zero,
yet a query with them still excludes every product that lacks them:

In [14]:
idf = pd.Series(engine.idf).sort_values()
idf.head(10).round(3).rename("idf").to_frame()

,idf
wear,0.300
wash,0.317
cotton,0.345
western,0.385
sleev,0.443
machin,0.581
regular,0.603
neck,0.690
women,0.710
blend,0.727
